# gVXR polychromatic package radiography

This reduced Colab experiment compares unfiltered and Al/Cu-filtered 160 kV spectra on a deterministic SiO₂/Cu/SAC305 package phantom. It validates spectral simulation and records reproducibility metadata; it is not a calibrated scanner model.

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !apt-get update -qq
    !apt-get install -y -qq libnvidia-gl-580 libxcb-res0 libxcb-ewmh2 libxcb-composite0 libxcb-cursor0 libxcb-xinerama0 libxcb-keysyms1 libxcb-icccm4 libxcb-xkb1

repo = Path('/content/xsim-chip')
if repo.exists():
    !git -C {repo} pull --ff-only
else:
    !git clone -q https://github.com/yuweimin2077-hub/xsim-chip.git {repo}
%cd /content/xsim-chip
%pip install -q -e '.[spectral]'

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libnvidia-cfg1-580:amd64.
(Reading database ... 122579 files and directories currently installed.)
Preparing to unpack .../00-libnvidia-cfg1-580_580.178.04-1ubuntu1_amd64.deb ...
Unpacking libnvidia-cfg1-580:amd64 (580.178.04-1ubuntu1) ...
Selecting previously unselected package libnvidia-decode-580:amd64.
Preparing to unpack .../01-libnvidia-decode-580_580.178.04-1ubuntu1_amd64.deb ...
Unpacking libnvidia-decode-580:amd64 (580.178.04-1ubuntu1) ...
Selecting previously unselected package libnvidia-gpucomp-580:amd64.
Preparing to unpack .../02-libnvidia-gpucomp-580_580.178.04-1ubuntu1_amd64.deb ...
Unpacking libnvidia-gpucomp-580:amd64 (580.178.04-1ubuntu1) ...
Selecting previously unselected package nvidia-persistenced.
Preparing to unpack .../03-nvidia-persist

In [2]:
import json
import time
import matplotlib.pyplot as plt
import numpy as np
from gvxrPython3 import gvxr
from xsim_chip_analysis import (
    ConeBeamConfig, SimulationConfig, SpectrumConfig, build_run_manifest,
    normalise_energy_image, summarise_spectrum,
)

spectrum = SpectrumConfig()
geometry = ConeBeamConfig()
print('Core gVXR:', gvxr.getVersionOfCoreGVXR())
print('SimpleGVXR:', gvxr.getVersionOfSimpleGVXR())
print('Spectrum config:', json.dumps(spectrum.to_manifest(), indent=2))
print('Cone-beam config:', json.dumps(geometry.to_manifest(), indent=2))

Core gVXR: gVirtualXRay core library (gvxr) 2.1.0 (2026-06-09T15:18:01) [Compiler: GNU g++] on Linux
SimpleGVXR: SimpleGVXR 2.1.0 (2026-06-09T15:18:02) [Compiler: GNU g++] on Linux
Spectrum config: {
  "tube_voltage_kv": 160.0,
  "energy_bin_size_kev": 4.0,
  "exposure_mas": 1.0,
  "filters_mm": [
    {
      "element": "Al",
      "thickness_mm": 0.5
    },
    {
      "element": "Cu",
      "thickness_mm": 1.0
    }
  ]
}
Cone-beam config: {
  "source_position_mm": [
    -200.0,
    0.0,
    0.0
  ],
  "detector_position_mm": [
    400.0,
    0.0,
    0.0
  ],
  "detector_pixels_xy": [
    192,
    128
  ],
  "detector_pixel_size_mm": [
    0.16,
    0.16
  ],
  "magnification": 3.0,
  "object_pixel_size_mm": [
    0.05333333333333334,
    0.05333333333333334
  ]
}


## Compact multi-material package

The geometry uses non-overlapping built-in cuboids to avoid thousands of upstream STL components while preserving the three material classes and cone-beam magnification.

In [3]:
started = time.perf_counter()
gvxr.createOpenGLContext()
gvxr.setSourcePosition(*geometry.source_position_mm, 'mm')
gvxr.usePointSource()
gvxr.setDetectorPosition(*geometry.detector_position_mm, 'mm')
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(*geometry.detector_pixels_xy)
gvxr.setDetectorPixelSize(*geometry.detector_pixel_size_mm, 'mm')

def add_cuboid(label, size_xyz_mm, position_xyz_mm):
    gvxr.makeCuboid(label, *size_xyz_mm, 'mm')
    gvxr.addPolygonMeshAsOuterSurface(label)
    gvxr.translateNode(label, *position_xyz_mm, 'mm')

add_cuboid('SiO2_core', (0.8, 8.0, 5.0), (0.0, 0.0, 0.0))
gvxr.setMixture('SiO2_core', [14, 8], [0.467, 0.533])
gvxr.setDensity('SiO2_core', 2.20, 'g/cm3')

for index, z_mm in enumerate((-1.4, 0.0, 1.4)):
    label = f'Cu_trace_{index}'
    add_cuboid(label, (0.12, 6.8, 0.18), (0.46, 0.0, z_mm))
    gvxr.setElement(label, 'Cu')

for index, y_mm in enumerate((-2.4, 0.0, 2.4)):
    label = f'SAC305_joint_{index}'
    add_cuboid(label, (0.55, 0.8, 0.8), (0.795, y_mm, -1.9))
    gvxr.setMixture(label, [50, 47, 29], [0.965, 0.030, 0.005])
    gvxr.setDensity(label, 7.38, 'g/cm3')

print('Package primitives created: 1 SiO2 core, 3 Cu traces, 3 SAC305 joints')

Package primitives created: 1 SiO2 core, 3 Cu traces, 3 SAC305 joints


## Unfiltered versus filtered polychromatic exposure

gVXR generates the tube spectrum and applies material-dependent Beer–Lambert attenuation. The filtered exposure adds 0.5 mm Al inherent filtration and 1.0 mm Cu filtration.

In [4]:
gvxr.setEnergyBinSize(spectrum.energy_bin_size_kev, 'keV')
gvxr.setVoltage(spectrum.tube_voltage_kv, 'kV')
gvxr.setmAs(spectrum.exposure_mas)

energy_unfiltered = np.asarray(gvxr.getEnergyBins('keV'), dtype=np.float32)
counts_unfiltered = np.asarray(gvxr.getPhotonCountsPerPixelAtSDD(), dtype=np.float64)
image_unfiltered = normalise_energy_image(
    np.asarray(gvxr.computeXRayImage()), gvxr.getTotalEnergyWithDetectorResponse()
)
summary_unfiltered = summarise_spectrum(energy_unfiltered, counts_unfiltered)

gvxr.addInherentFilter(spectrum.filters_mm[0][0], spectrum.filters_mm[0][1], 'mm')
gvxr.addFilter(spectrum.filters_mm[1][0], spectrum.filters_mm[1][1], 'mm')
energy_filtered = np.asarray(gvxr.getEnergyBins('keV'), dtype=np.float32)
counts_filtered = np.asarray(gvxr.getPhotonCountsPerPixelAtSDD(), dtype=np.float64)
image_filtered = normalise_energy_image(
    np.asarray(gvxr.computeXRayImage()), gvxr.getTotalEnergyWithDetectorResponse()
)
summary_filtered = summarise_spectrum(energy_filtered, counts_filtered)

object_mask = np.minimum(image_unfiltered, image_filtered) < 0.999
metrics = {
    'mean_energy_shift_kev': summary_filtered['mean_energy_kev'] - summary_unfiltered['mean_energy_kev'],
    'mean_unfiltered_transmission': float(image_unfiltered[object_mask].mean()),
    'mean_filtered_transmission': float(image_filtered[object_mask].mean()),
    'mean_absolute_transmission_change': float(np.abs(image_filtered - image_unfiltered)[object_mask].mean()),
    'object_pixels': int(object_mask.sum()),
}
print('Unfiltered:', json.dumps(summary_unfiltered, indent=2))
print('Filtered:', json.dumps(summary_filtered, indent=2))
print('Comparison:', json.dumps(metrics, indent=2))

ValueError: image must be a finite, non-negative 2D array

In [5]:
raw = np.asarray(gvxr.computeXRayImage())
print({'shape': raw.shape, 'dtype': str(raw.dtype), 'min': float(np.nanmin(raw)), 'max': float(np.nanmax(raw)), 'nan': int(np.isnan(raw).sum()), 'negative': int((raw < 0).sum())})

ValueError: zero-size array to reduction operation fmin which has no identity

In [6]:
raw = np.asarray(gvxr.computeXRayImage())
print({'detector': gvxr.getDetectorNumberOfPixels(), 'energy_bins': len(gvxr.getEnergyBins('keV')), 'photon_bins': len(gvxr.getPhotonCountsPerPixelAtSDD()), 'total_energy': gvxr.getTotalEnergyWithDetectorResponse(), 'raw_shape': raw.shape, 'raw_size': raw.size})

{'detector': (192, 128), 'energy_bins': 40, 'photon_bins': 40, 'total_energy': 149843.40625, 'raw_shape': (0,), 'raw_size': 0}


In [7]:
print(gvxr.createNewContext.__doc__)
print([name for name in dir(gvxr) if 'context' in name.lower() or 'renderer' in name.lower() or 'opengl' in name.lower()])




Create an OpenGL context.  

Parameters
----------
* `aVisibilityFlag` :  
    flag controlling if the window should be visible (1) or hidden (0). (default
    value: 0)  
* `aRenderer` :  
    Select the renderer to use, e.g. OpenGL or Vulkan. (default value: OPENGL)  
* `aWindowID` :  
    the numerical ID of the context to create (default value: -1, means that the
    ID will be automatically generated)  
* `aRendererMajorVersion` :  
    Select the major version of the renderer. (default value: 4)  
* `aRendererMinorVersion` :  
    Select the minor version of the renderer. (default value: 3)  
* `aMixedPrecisionFlag` :  
    Set of mixed-precision flag to select the number of bits per floating-point
    numbers in GPU memory for various textures. See computeMixedPrecisionFlag
    for help about how to set the flag. (default value: 255, i.e. FP32 for all
    textures)  


['autoCreateOpenGLContext', 'createNewContext', 'createOpenGLContext', 'getOpenGlRenderer', 'getOpenGlVendor

In [8]:
print({'vendor': gvxr.getOpenGlVendor(), 'renderer': gvxr.getOpenGlRenderer(), 'version': gvxr.getOpenGlVersion()})

{'vendor': 'NVIDIA Corporation', 'renderer': 'Tesla T4/PCIe/SSE2', 'version': '4.3.0 NVIDIA 580.82.07'}


In [9]:
gvxr.setMonoChromaticPerPixelAtSDD(80, 'keV', 1000)
mono = np.asarray(gvxr.computeXRayImage())
print({'mono_shape': mono.shape, 'mono_size': mono.size})

{'mono_shape': (0,), 'mono_size': 0}


In [10]:
gvxr.terminate()
gvxr.createNewContext()
gvxr.setSourcePosition(-20.0, 0.0, 0.0, 'cm')
gvxr.usePointSource()
gvxr.setMonoChromaticPerPixelAtSDD(80, 'keV', 1000)
gvxr.setDetectorPosition(20.0, 0.0, 0.0, 'cm')
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(640, 320)
gvxr.setDetectorPixelSize(0.5, 0.5, 'mm')
gvxr.makeCuboid('Test', 5, 4, 3, 'cm')
gvxr.addPolygonMeshAsOuterSurface('Test')
gvxr.setCompound('Test', 'H2O')
gvxr.setDensity('Test', 1.0, 'g/cm3')
test_image = np.asarray(gvxr.computeXRayImage())
print({'renderer': gvxr.getOpenGlRenderer(), 'shape': test_image.shape, 'size': test_image.size})

{'renderer': 'Tesla T4/PCIe/SSE2', 'shape': (320, 640), 'size': 204800}


In [ ]:
probability_unfiltered = counts_unfiltered / counts_unfiltered.sum()
probability_filtered = counts_filtered / counts_filtered.sum()
figure, axes = plt.subplots(2, 2, figsize=(13, 9))
axes[0, 0].bar(energy_unfiltered, probability_unfiltered, width=spectrum.energy_bin_size_kev, alpha=0.65, label='Unfiltered')
axes[0, 0].bar(energy_filtered, probability_filtered, width=spectrum.energy_bin_size_kev, alpha=0.65, label='0.5 mm Al + 1.0 mm Cu')
axes[0, 0].set(xlabel='Energy (keV)', ylabel='Photon probability', title='160 kV spectra')
axes[0, 0].legend()
axes[0, 1].imshow(image_unfiltered, cmap='gray', vmin=0, vmax=1)
axes[0, 1].set_title('Unfiltered transmission')
axes[1, 0].imshow(image_filtered, cmap='gray', vmin=0, vmax=1)
axes[1, 0].set_title('Filtered transmission')
difference = image_filtered - image_unfiltered
limit = max(float(np.abs(difference).max()), 1e-6)
axes[1, 1].imshow(difference, cmap='coolwarm', vmin=-limit, vmax=limit)
axes[1, 1].set_title('Filtered − unfiltered')
for ax in (axes[0, 1], axes[1, 0], axes[1, 1]): ax.axis('off')
plt.tight_layout()

## Save reproducibility artefacts

The compact arrays and manifest are written outside Git. The manifest distinguishes measured outputs from scanner assumptions and limitations.

In [ ]:
elapsed = time.perf_counter() - started
output_dir = Path('/content/xsim_outputs/gvxr_spectral')
output_dir.mkdir(parents=True, exist_ok=True)
manifest = build_run_manifest(SimulationConfig.quick_colab(), elapsed_seconds=elapsed)
manifest['experiment'] = 'gvxr-polychromatic-radiography-v1'
manifest['spectrum'] = spectrum.to_manifest()
manifest['geometry'] = geometry.to_manifest()
manifest['gvxr_runtime'] = {
    'core': gvxr.getVersionOfCoreGVXR(),
    'simple': gvxr.getVersionOfSimpleGVXR(),
}
manifest['result'] = {
    'unfiltered_spectrum': summary_unfiltered,
    'filtered_spectrum': summary_filtered,
    **metrics,
}
manifest['limitations'] = [
    'Compact cuboid phantom, not the full NIST STL package',
    'Ideal energy-integrating detector; no scatter or detector blur',
    'Geometry and exposure are simulation assumptions, not scanner calibration',
]
(output_dir / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
np.savez_compressed(
    output_dir / 'spectral_radiography.npz',
    energy_unfiltered_kev=energy_unfiltered, counts_unfiltered=counts_unfiltered,
    energy_filtered_kev=energy_filtered, counts_filtered=counts_filtered,
    image_unfiltered=image_unfiltered, image_filtered=image_filtered,
)
figure.savefig(output_dir / 'spectral_comparison.png', dpi=160, bbox_inches='tight')
gvxr.terminate()
print(json.dumps(manifest, indent=2))
print('Saved to', output_dir)